This example requires the following dependencies to be installed:
pip install lightly[timm]

In [ ]:
!pip install lightly[timm]

Note: The model and training settings do not follow the reference settings
from the paper. The settings are chosen such that the example can easily be
run on a small dataset with a single GPU.

In [ ]:
import torch
import torchvision
from timm.models.vision_transformer import vit_small_patch16_224
from torch import Tensor
from torch.nn import Module
from torch.optim import AdamW

In [ ]:
from lightly.loss import LeJEPALoss
from lightly.models.modules import LeJEPAProjectionHead
from lightly.transforms.dino_transform import DINOTransform

In [ ]:
def _get_backbone_output_dim(backbone: Module) -> int:
    """Get the output dimension of a backbone by passing a dummy input through it."""
    with torch.inference_mode():
        dummy_input = torch.zeros(1, 3, 224, 224)
        output = backbone(dummy_input)
        output_dim = output.shape[1]
    return output_dim

In [ ]:
class LeJEPA(Module):
    def __init__(
        self,
    ) -> None:
        super().__init__()

        self.backbone = vit_small_patch16_224(
            pretrained=False,
            pos_embed="learn",
            num_classes=0,
            dynamic_img_size=True,
            drop_path_rate=0.1,
        )

        backbone_out_dims = _get_backbone_output_dim(self.backbone)
        self.projection_head = LeJEPAProjectionHead(input_dim=backbone_out_dims)

    def forward(self, x: Tensor) -> Tensor:
        emb = self.backbone(x)
        proj = self.projection_head(emb)
        return proj

In [ ]:
model = LeJEPA()

In [ ]:
transform = DINOTransform(
    global_crop_scale=(0.3, 1),
    local_crop_scale=(0.05, 0.3),
    gaussian_blur=(0.5, 0.5, 0.5),
    n_local_views=6,
)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
dataset = torchvision.datasets.VOCDetection(
    "datasets/pascal_voc",
    download=True,
    transform=transform,
)
# Or create a dataset from a folder containing images or videos.
# dataset = LightlyDataset("path/to/folder")

In [ ]:
dataloader = torch.utils.data.DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
    drop_last=True,
    num_workers=0,
)

In [ ]:
# Create the loss functions.
lejepa_criterion = LeJEPALoss()

In [ ]:
# Move loss to correct device because it also contains parameters.
lejepa_criterion = lejepa_criterion.to(device)

In [ ]:
optimizer = AdamW(model.parameters(), lr=5e-4, weight_decay=5e-2)

In [ ]:
epochs = 50
num_batches = len(dataloader)

In [ ]:
print("Starting Training")
for epoch in range(epochs):
    total_loss = 0
    for batch_idx, batch in enumerate(dataloader):
        views = batch[0]
        views = [view.to(device) for view in views]
        global_views = views[:2]
        local_views = views[2:]

        global_proj = torch.stack([model(view) for view in global_views])
        local_proj = torch.stack([model(view) for view in local_views])

        loss = lejepa_criterion(local_proj=local_proj, global_proj=global_proj)
        total_loss += loss.detach()
        print(
            f"Epoch [{epoch + 1}/{epochs}] "
            f"Batch [{batch_idx + 1}/{num_batches}] "
            f"Loss [{loss.item():.4f}]"
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    avg_loss = total_loss / num_batches
    print(f"epoch: {epoch:>02}, loss: {avg_loss:.5f}")